# Extract layers for step 1

Every story in the Impact Sphere includes a Step 1 layer — a national or regional overview that provides geographic context for the story. This layer always covers the full country or region, even when the story itself focuses on a specific city or area within it (e.g., a story about Nairobi still uses a Kenya-wide Step 1 layer; a story about West Africa uses a regional mosaic).

This notebook extracts Step 1 layers from one of four global datasets:
- **WorldCover** — ESA global land cover map (10 m)
- **WSF Building Area** — DLR World Settlement Footprint 3D building area
- **Global Surface Water (GSW)** — JRC surface water occurrence
- **WorldCereal** — ESA global cropland classification

More sources may be added in the future.

## Workflows

Each data source section provides two workflows:
- **Single country** — clip or download the dataset for one country, then style and process
- **Multi-country region** — clip or download per country (each clipped to its GADM boundary), merge into a regional mosaic, then style and process

## Setup

### Library import

In [ ]:
import os
import sys
from glob import glob
from pathlib import Path

import rasterio as rio
from rasterio.merge import merge

sys.path.append("../src")

from data_processing.download_step1 import download_worldcover_for_country, download_gsw_for_country
from data_processing.raster_processor import RasterProcessor
from data_processing.utils import clip_raster_to_country_and_create_cog

## Data retrieval and processing

Select the section for the data source used in the story. Edit the working cells with the target country or region and run the corresponding workflow.

### [WorldCover](https://github.com/ESA-WorldCover/esa-worldcover-datasets)

#### Single country

In [ ]:
# WORKING CELL — set country, run, then revert.
country = "your_country"

# Download tiles overlapping the country and merge them
clipped_raster = download_worldcover_for_country(country, target_resolution=50)

In [ ]:
# Style and process raster
print(f"Processing WorldCover for {country}...")
BASE_PATH = Path("../data/processed/WorldCover/")
input_file = BASE_PATH / f"Mosaic/WorldCover_2021_{country.replace(' ', '_')}_Clipped.tif"
style_file = BASE_PATH / "WorldCover.qml"
output_file = BASE_PATH / f"Rasters/WorldCover_2021_{country.replace(' ', '_')}.tif"
layer_name = f"{country.replace(' ', '_')}_WorldCover_2021"

RasterProcessor(
    input_file,
    output_file=output_file,
    qml_file=style_file,
    layer_name=layer_name,
    upload=True,
    create_mbtiles=True,
    max_zoom=7,
).process()

#### Multi-country region

In [ ]:
# WORKING CELL — set countries and region, run, then revert.
countries = [
    "country_1",
    "country_2",
]
region = "your_region"

# Download and clip for each country
for country in countries:
    print(f"Downloading WorldCover for {country}...")
    download_worldcover_for_country(country, target_resolution=50)

In [ ]:
# Merge per-country rasters into a regional mosaic
BASE_PATH = Path("../data/processed/WorldCover/")
raster_dir = BASE_PATH / "Mosaic/"

raster_files = []
for country in countries:
    pattern = str(raster_dir / f"WorldCover_2021_{country.replace(' ', '_')}_Clipped.tif")
    raster_files.extend(glob(pattern))

raster_files = list(dict.fromkeys(raster_files))
print(f"Found {len(raster_files)} raster files to merge.")

if not raster_files:
    raise FileNotFoundError("No matching raster files found. Check that per-country downloads completed.")

src_files_to_mosaic = [rio.open(fp) for fp in raster_files]
mosaic, out_trans = merge(src_files_to_mosaic)

out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans,
    "compress": "DEFLATE",
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
})

output_path = raster_dir / f"WorldCover_2021_{region}_Clipped.tif"
with rio.open(output_path, "w", **out_meta) as dest:
    dest.write(mosaic)

print(f"Merged raster saved to: {output_path}")

In [ ]:
# Style and process regional mosaic
print(f"Processing WorldCover for {region}...")
BASE_PATH = Path("../data/processed/WorldCover/")
input_file = BASE_PATH / f"Mosaic/WorldCover_2021_{region}_Clipped.tif"
style_file = BASE_PATH / "WorldCover.qml"
output_file = BASE_PATH / f"Rasters/WorldCover_2021_{region}.tif"
layer_name = f"{region}_WorldCover_2021"

RasterProcessor(
    input_file,
    output_file=output_file,
    qml_file=style_file,
    layer_name=layer_name,
    upload=True,
    create_mbtiles=True,
    max_zoom=7,
).process()

### [World Settlement Footprint (WSF) Building Area](https://download.geoservice.dlr.de/WSF3D/files/global/WSF3D_V02_BuildingArea.tif)

#### Single country

In [ ]:
# WORKING CELL — set country, run, then revert.
country = "your_country"

# Clip WSF Building Area raster to country and create COG
raster_file = "../data/processed/WSF/WSF3D_V02_BuildingArea.tif"
output_dir = "../data/processed/WSF/Clipped/"

cog_file = clip_raster_to_country_and_create_cog(raster_file, country, output_dir)
print(f"Created COG: {cog_file}")

In [ ]:
# Style and process raster
print(f"Processing WSF Building Area for {country}...")
BASE_PATH = Path("../data/processed/WSF/")
input_file = BASE_PATH / f"Clipped/WSF3D_V02_BuildingArea_{country.replace(' ', '_')}.tif"
style_file = BASE_PATH / "WSF3D_V02_BuildingArea_satellite.qml"
output_file = BASE_PATH / f"Rasters/WSF3D_V02_BuildingArea_{country.replace(' ', '_')}.tif"
layer_name = f"{country.replace(' ', '_')}_WSF3D_BuildingArea"

RasterProcessor(
    input_file,
    output_file=output_file,
    qml_file=style_file,
    layer_name=layer_name,
    upload=False,
    create_mbtiles=False,
).process()

#### Multi-country region

In [ ]:
# WORKING CELL — set countries and region, run, then revert.
countries = [
    "country_1",
    "country_2",
]
region = "your_region"

# Clip WSF to each country
raster_file = "../data/processed/WSF/WSF3D_V02_BuildingArea.tif"
output_dir = "../data/processed/WSF/Clipped/"

cog_files = []
for country in countries:
    cog_file = clip_raster_to_country_and_create_cog(raster_file, country, output_dir)
    if cog_file:
        cog_files.append(cog_file)
print(f"Created {len(cog_files)} COG files.")

In [ ]:
# Merge per-country COGs into a regional mosaic
raster_dir = "../data/processed/WSF/"

raster_files = []
for country in countries:
    pattern = os.path.join(raster_dir, f"*_{country.replace(' ', '_')}.tif")
    raster_files.extend(glob(pattern))

raster_files = list(dict.fromkeys(raster_files))
print(f"Found {len(raster_files)} raster files to merge.")

if not raster_files:
    raise FileNotFoundError("No matching raster files found. Check filenames and directory path.")

src_files_to_mosaic = [rio.open(fp) for fp in raster_files]
mosaic, out_trans = merge(src_files_to_mosaic)

out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans,
    "compress": "DEFLATE",
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
    "nodata": 0
})

output_path = os.path.join(raster_dir, f"WSF3D_V02_BuildingArea_{region}.tif")
with rio.open(output_path, "w", **out_meta) as dest:
    dest.write(mosaic)

print(f"Merged raster saved to: {output_path}")

In [ ]:
# Style and process regional mosaic
print(f"Processing WSF Building Area for {region}...")
BASE_PATH = Path("../data/processed/WSF/")
input_file = BASE_PATH / f"WSF3D_V02_BuildingArea_{region}.tif"
style_file = BASE_PATH / "WSF3D_V02_BuildingArea_satellite.qml"
output_file = BASE_PATH / f"Rasters/WSF3D_V02_BuildingArea_{region}.tif"
layer_name = f"{region}_WSF3D_BuildingArea"

RasterProcessor(
    input_file,
    output_file=output_file,
    qml_file=style_file,
    layer_name=layer_name,
    upload=False,
    create_mbtiles=False,
).process()

### [Global Surface Water](https://global-surface-water.appspot.com/)

#### Single country

In [ ]:
# WORKING CELL — set country, run, then revert.
country = "your_country"

clipped_raster = download_gsw_for_country(country)

In [ ]:
# Style and process raster
print(f"Processing GSW for {country}...")
BASE_PATH = Path("../data/processed/GSW/")
input_file = BASE_PATH / f"Mosaics/occurrence_{country.replace(' ', '_')}_clipped.tif"
style_file = BASE_PATH / "GSW.qml"
output_file = BASE_PATH / f"Rasters/occurrence_{country.replace(' ', '_')}.tif"
layer_name = f"{country.replace(' ', '_')}_GSW_Occurrence"

RasterProcessor(
    input_file,
    output_file=output_file,
    qml_file=style_file,
    layer_name=layer_name,
    upload=True,
    create_mbtiles=True,
    max_zoom=6,
).process()

#### Multi-country region

In [ ]:
# WORKING CELL — set countries and region, run, then revert.
countries = [
    "country_1",
    "country_2",
]
region = "your_region"

# Download and clip GSW for each country
for country in countries:
    clipped_raster = download_gsw_for_country(country)
    print(f"Downloaded GSW for {country}")

In [ ]:
# Merge per-country rasters into a regional mosaic
raster_dir = "../data/processed/GSW/Mosaics/"

raster_files = []
for country in countries:
    pattern = os.path.join(raster_dir, f"*_{country.replace(' ', '_')}_clipped.tif")
    raster_files.extend(glob(pattern))

raster_files = list(dict.fromkeys(raster_files))
print(f"Found {len(raster_files)} raster files to merge.")

if not raster_files:
    raise FileNotFoundError("No matching raster files found. Check filenames and directory path.")

src_files_to_mosaic = [rio.open(fp) for fp in raster_files]
mosaic, out_trans = merge(src_files_to_mosaic)

out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans,
    "compress": "DEFLATE",
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
    "nodata": 0
})

output_path = os.path.join(raster_dir, f"occurrence_{region}_clipped.tif")
with rio.open(output_path, "w", **out_meta) as dest:
    dest.write(mosaic)

print(f"Merged raster saved to: {output_path}")

In [ ]:
# Style and process regional mosaic
print(f"Processing GSW for {region}...")
BASE_PATH = Path("../data/processed/GSW/")
input_file = BASE_PATH / f"Mosaics/occurrence_{region}_clipped.tif"
style_file = BASE_PATH / "GSW.qml"
output_file = BASE_PATH / f"Rasters/occurrence_{region}.tif"
layer_name = f"{region}_GSW_Occurrence"

RasterProcessor(
    input_file,
    output_file=output_file,
    qml_file=style_file,
    layer_name=layer_name,
    upload=True,
    create_mbtiles=True,
    max_zoom=6,
).process()

### [WorldCereal](https://esa-worldcereal.org/en/products/global-maps)

#### Single country

In [ ]:
# WORKING CELL — set country, run, then revert.
country = "your_country"

# Clip raster to country and create COG
raster_file = "../data/processed/WorldCereal/2021_tc-annual_temporarycrops_classification_0.004deg.tif"
output_dir = "../data/processed/WorldCereal/Clipped/"

cog_file = clip_raster_to_country_and_create_cog(raster_file, country, output_dir=output_dir)
print(f"Created COG: {cog_file}")

In [ ]:
# Style and process raster
print(f"Processing WorldCereal for {country}...")
BASE_PATH = Path("../data/processed/WorldCereal/")
input_file = BASE_PATH / f"Clipped/2021_tc-annual_temporarycrops_classification_0.004deg_{country.replace(' ', '_')}.tif"
style_file = BASE_PATH / "temporarycrops.qml"
output_file = BASE_PATH / f"Rasters/WorldCereal_{country.replace(' ', '_')}.tif"
layer_name = f"{country.replace(' ', '_')}_WorldCereal"

RasterProcessor(
    input_file,
    output_file=output_file,
    qml_file=style_file,
    layer_name=layer_name,
    upload=True,
    create_mbtiles=True,
    max_zoom=8,
).process()

#### Multi-country region

In [ ]:
# WORKING CELL — set countries and region, run, then revert.
countries = [
    "country_1",
    "country_2",
]
region = "your_region"

# Clip WorldCereal to each country
raster_file = "../data/processed/WorldCereal/2021_tc-annual_temporarycrops_classification_0.004deg.tif"
output_dir = "../data/processed/WorldCereal/Clipped/"

cog_files = []
for country in countries:
    cog_file = clip_raster_to_country_and_create_cog(raster_file, country, output_dir=output_dir)
    if cog_file:
        cog_files.append(cog_file)
print(f"Created {len(cog_files)} COG files.")

In [ ]:
# Merge per-country COGs into a regional mosaic
raster_dir = "../data/processed/WorldCereal/Clipped/"

raster_files = []
for country in countries:
    pattern = os.path.join(raster_dir, f"*_{country.replace(' ', '_')}.tif")
    raster_files.extend(glob(pattern))

raster_files = list(dict.fromkeys(raster_files))
print(f"Found {len(raster_files)} raster files to merge.")

if not raster_files:
    raise FileNotFoundError("No matching raster files found. Check filenames and directory path.")

src_files_to_mosaic = [rio.open(fp) for fp in raster_files]
mosaic, out_trans = merge(src_files_to_mosaic)

out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans,
    "compress": "DEFLATE",
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
    "nodata": 0
})

output_path = os.path.join(
    raster_dir,
    f"2021_tc-annual_temporarycrops_classification_0.004deg_{region}.tif"
)
with rio.open(output_path, "w", **out_meta) as dest:
    dest.write(mosaic)

print(f"Merged raster saved to: {output_path}")

In [ ]:
# Style and process regional mosaic
print(f"Processing WorldCereal for {region}...")
BASE_PATH = Path("../data/processed/WorldCereal/")
input_file = BASE_PATH / f"Clipped/2021_tc-annual_temporarycrops_classification_0.004deg_{region}.tif"
style_file = BASE_PATH / "temporarycrops.qml"
output_file = BASE_PATH / f"Rasters/WorldCereal_{region}.tif"
layer_name = f"{region}_WorldCereal"

RasterProcessor(
    input_file,
    output_file=output_file,
    qml_file=style_file,
    layer_name=layer_name,
    upload=True,
    create_mbtiles=True,
    max_zoom=8,
).process()